# Google Play Phase 2 Cadence Test — Run A

This notebook starts the cadence / operating assumption testing after the Day 1–Day 3 controlled repeated-run baseline.

The goal is to compare whether a same-day second collection behaves differently from the once-daily controlled repeated baseline.

This run keeps the core setup unchanged:

- same 10 apps
- same 1,200-review target per app
- same Google Play source
- same language and country setting
- same Phase 2 SQLite database
- same duplicate prevention logic

The new operating assumption being tested is:

**once-daily controlled baseline vs. twice-daily same-day second collection**

This helps evaluate duplicate rate, new review capture, runtime, database growth, app-level behavior, and quality-flag patterns under a higher collection frequency.

In [1]:
!pip -q install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00


## 1. Import packages and set run configuration

This notebook continues from the fixed Phase 2 Day 3 database.

Expected database tables:

- `phase2_reviews_raw`
- `phase2_reviews_cleaned`
- `phase2_apps`
- `phase2_ingestion_runs`
- `phase2_app_run_summary`
- `phase2_quality_flags`

In [2]:
import os
import re
import gc
import json
import time
import shutil
import sqlite3
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from google_play_scraper import reviews, Sort

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATABASE_DIR = BASE_DIR / "database"
OUTPUT_DIR = BASE_DIR / "outputs"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / "google_play_reviews.sqlite"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
SORT_METHOD = Sort.NEWEST

TARGET_REVIEWS_PER_APP = 1200
REQUEST_SLEEP_SECONDS = 2

RUN_LABEL = "phase2_cadence_runA_twice_daily_test"
FREQUENCY_LABEL = "twice_daily_same_day_second_run"
RUN_ID = f"{RUN_LABEL}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print("Run ID:", RUN_ID)
print("Run label:", RUN_LABEL)
print("Frequency label:", FREQUENCY_LABEL)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Run ID: phase2_cadence_runA_twice_daily_test_20260709_210858
Run label: phase2_cadence_runA_twice_daily_test
Frequency label: twice_daily_same_day_second_run
Database path: /content/database/google_play_reviews.sqlite
Output folder: /content/outputs


## 2. Upload the fixed Day 3 database

Upload either:

- the fixed Day 3 GitHub upload zip, or
- the compressed SQLite file after Day 3.

The valid database should already include Day 1, Day 2, and Day 3 run history.

In [4]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile
import sqlite3
import pandas as pd

UPLOAD_DIR = Path("/content/uploaded_day3_files")

# Clean old failed upload/extract folders first
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Please upload the fixed Day 3 GitHub zip or the compressed Day 3 SQLite database.")
print("Recommended file: phase2_day3_github_upload_files_20260709_040822_utc_FIXED.zip")
print("Also accepted: google_play_reviews_after_day3.sqlite.zip")

uploaded = files.upload()

uploaded_paths = []

for file_name in uploaded.keys():
    src = Path("/content") / file_name
    dst = UPLOAD_DIR / file_name

    if dst.exists():
        dst.unlink()

    shutil.move(str(src), str(dst))
    uploaded_paths.append(dst)
    print("Uploaded:", dst)

# Unzip uploaded zip files
for uploaded_path in uploaded_paths:
    if uploaded_path.is_file() and uploaded_path.suffix.lower() == ".zip":
        # Important: do NOT create an extraction folder ending with .sqlite
        extract_dir = UPLOAD_DIR / f"{uploaded_path.stem}_unzipped"
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(uploaded_path, "r") as zf:
            zf.extractall(extract_dir)

        print("Unzipped:", uploaded_path.name, "->", extract_dir)

# Unzip nested zip files, but only real files
nested_zip_files = [
    p for p in UPLOAD_DIR.rglob("*.zip")
    if p.is_file()
]

for nested_zip in nested_zip_files:
    # Skip the original uploaded zip because it was already extracted above
    if nested_zip in uploaded_paths:
        continue

    nested_extract_dir = nested_zip.parent / f"{nested_zip.stem}_unzipped"
    nested_extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(nested_zip, "r") as zf:
            zf.extractall(nested_extract_dir)
        print("Unzipped nested zip:", nested_zip.name, "->", nested_extract_dir)
    except Exception as e:
        print("Skipped nested zip:", nested_zip.name, e)

# Find SQLite candidates: only actual files, not folders
db_candidates = []

for pattern in ["*.sqlite", "*.sqlite3", "*.db"]:
    db_candidates.extend([
        p for p in UPLOAD_DIR.rglob(pattern)
        if p.is_file()
    ])

if not db_candidates:
    raise FileNotFoundError(
        "No SQLite database file found. Please upload the fixed Day 3 zip or the Day 3 SQLite database."
    )

def inspect_phase2_db(db_path):
    try:
        with sqlite3.connect(str(db_path)) as test_conn:
            tables = pd.read_sql_query("""
                SELECT name
                FROM sqlite_master
                WHERE type='table'
            """, test_conn)["name"].tolist()

            required_tables = [
                "phase2_reviews_raw",
                "phase2_reviews_cleaned",
                "phase2_apps",
                "phase2_ingestion_runs",
                "phase2_app_run_summary",
                "phase2_quality_flags",
            ]

            missing_tables = [t for t in required_tables if t not in tables]

            if missing_tables:
                return {
                    "path": str(db_path),
                    "size_mb": float(db_path.stat().st_size / (1024 * 1024)),
                    "valid_day3_database": False,
                    "reason": f"Missing tables: {missing_tables}",
                    "raw_rows": None,
                    "cleaned_rows": None,
                    "app_count": None,
                    "run_count": None,
                    "target_reviews_per_app": None,
                }

            raw_rows = int(pd.read_sql_query(
                "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
                test_conn
            )["n"].iloc[0])

            cleaned_rows = int(pd.read_sql_query(
                "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
                test_conn
            )["n"].iloc[0])

            app_count = int(pd.read_sql_query(
                "SELECT COUNT(*) AS n FROM phase2_apps",
                test_conn
            )["n"].iloc[0])

            run_count = int(pd.read_sql_query(
                "SELECT COUNT(*) AS n FROM phase2_ingestion_runs",
                test_conn
            )["n"].iloc[0])

            target_reviews = int(pd.read_sql_query("""
                SELECT target_reviews_per_app
                FROM phase2_ingestion_runs
                ORDER BY run_started_at DESC
                LIMIT 1
            """, test_conn)["target_reviews_per_app"].iloc[0])

            day3_count = int(pd.read_sql_query("""
                SELECT COUNT(*) AS n
                FROM phase2_ingestion_runs
                WHERE run_label = 'phase2_day3_controlled_repeated_run'
            """, test_conn)["n"].iloc[0])

            valid = (
                raw_rows >= 17000
                and cleaned_rows >= 17000
                and app_count == 10
                and run_count >= 3
                and target_reviews == 1200
                and day3_count >= 1
            )

            return {
                "path": str(db_path),
                "size_mb": float(db_path.stat().st_size / (1024 * 1024)),
                "valid_day3_database": valid,
                "reason": "valid" if valid else "Day 3 continuation checks did not pass",
                "raw_rows": raw_rows,
                "cleaned_rows": cleaned_rows,
                "app_count": app_count,
                "run_count": run_count,
                "target_reviews_per_app": target_reviews,
            }

    except Exception as e:
        return {
            "path": str(db_path),
            "size_mb": float(db_path.stat().st_size / (1024 * 1024)) if db_path.exists() else None,
            "valid_day3_database": False,
            "reason": f"Could not open or inspect database: {repr(e)}",
            "raw_rows": None,
            "cleaned_rows": None,
            "app_count": None,
            "run_count": None,
            "target_reviews_per_app": None,
        }

candidate_report = pd.DataFrame([inspect_phase2_db(p) for p in db_candidates])
display(candidate_report)

valid_candidates = candidate_report[
    candidate_report["valid_day3_database"] == True
].copy()

if valid_candidates.empty:
    raise ValueError(
        "No valid fixed Day 3 database found. "
        "The correct database should have phase2_* tables, 10 apps, at least 3 prior runs, "
        "at least 17,000 review rows, and 1,200 target reviews per app."
    )

valid_candidates = valid_candidates.sort_values("size_mb", ascending=False)
selected_db = Path(valid_candidates.iloc[0]["path"])

if DB_PATH.exists():
    DB_PATH.unlink()

shutil.copy2(selected_db, DB_PATH)

print("Selected valid Day 3 database:")
print(selected_db)
print("\nCopied to:")
print(DB_PATH)
print(f"Database size: {DB_PATH.stat().st_size / (1024 * 1024):.2f} MB")

Please upload the fixed Day 3 GitHub zip or the compressed Day 3 SQLite database.
Recommended file: phase2_day3_github_upload_files_20260709_040822_utc_FIXED.zip
Also accepted: google_play_reviews_after_day3.sqlite.zip


Saving google_play_reviews_after_day3.sqlite.zip to google_play_reviews_after_day3.sqlite.zip
Uploaded: /content/uploaded_day3_files/google_play_reviews_after_day3.sqlite.zip
Unzipped: google_play_reviews_after_day3.sqlite.zip -> /content/uploaded_day3_files/google_play_reviews_after_day3.sqlite_unzipped


,path,size_mb,valid_day3_database,reason,raw_rows,cleaned_rows,app_count,run_count,target_reviews_per_app
0,/content/uploaded_day3_files/google_play_revie...,43.761719,True,valid,17815,17815,10,3,1200


Selected valid Day 3 database:
/content/uploaded_day3_files/google_play_reviews_after_day3.sqlite_unzipped/google_play_reviews.sqlite

Copied to:
/content/database/google_play_reviews.sqlite
Database size: 43.76 MB


## 3. Connect to the database

This confirms that Run A continues from the Day 3 database and does not start a new database.

In [5]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

db_size_before_mb = float(DB_PATH.stat().st_size / (1024 * 1024))

raw_rows_before = int(pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
    conn
)["n"].iloc[0])

cleaned_rows_before = int(pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
    conn
)["n"].iloc[0])

prior_run_count = int(pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_ingestion_runs",
    conn
)["n"].iloc[0])

print("Connected to fixed Day 3 Phase 2 database.")
print("Database size before Run A:", round(db_size_before_mb, 2), "MB")
print("Raw review rows before Run A:", raw_rows_before)
print("Cleaned review rows before Run A:", cleaned_rows_before)
print("Prior Phase 2 runs:", prior_run_count)

Connected to fixed Day 3 Phase 2 database.
Database size before Run A: 43.76 MB
Raw review rows before Run A: 17815
Cleaned review rows before Run A: 17815
Prior Phase 2 runs: 3


## 4. Confirm baseline run history

Before cadence testing, the database should contain the Day 1, Day 2, and Day 3 controlled repeated runs.

In [6]:
baseline_runs_df = pd.read_sql_query("""
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(duplicates_skipped_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS duplicate_rate,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(new_records_inserted_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS new_insert_rate,
        errors_total,
        quality_flag_total,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_before,
        review_rows_after,
        review_rows_growth
    FROM phase2_ingestion_runs
    WHERE run_label LIKE 'phase2_day%'
    ORDER BY run_started_at
""", conn)

display(baseline_runs_df)

if len(baseline_runs_df) < 3:
    raise ValueError("Expected Day 1, Day 2, and Day 3 baseline runs before cadence testing.")

if int(baseline_runs_df["target_reviews_per_app"].iloc[-1]) != 1200:
    raise ValueError("The latest baseline run does not use the 1,200-review target.")

print("Day 1-Day 3 baseline history confirmed.")

,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,...,duplicate_rate,new_insert_rate,errors_total,quality_flag_total,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,1200,10,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.271117,completed,12000,...,0.0000,1.0000,0,12633,1.832031,25.492188,23.660156,0,12000,12000
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,1200,10,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.799585,completed,12000,...,0.9870,0.0130,0,12638,25.492188,30.187500,4.695312,12000,12156,156
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,1200,10,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,29.269241,completed,12000,...,0.5284,0.4716,0,12696,30.187500,43.761719,13.574219,12156,17815,5659


Day 1-Day 3 baseline history confirmed.


## 5. Load the same 10 apps

The app list is read directly from `phase2_apps`. No fallback list is used.

In [7]:
app_config_df = pd.read_sql_query("""
    SELECT
        app_name,
        app_id,
        source,
        language,
        country,
        title_from_store,
        score_from_store,
        ratings_from_store,
        installs_from_store
    FROM phase2_apps
    ORDER BY rowid
""", conn)

display(app_config_df)

if len(app_config_df) != 10:
    raise ValueError(f"Expected exactly 10 apps from phase2_apps, but found {len(app_config_df)}.")

if TARGET_REVIEWS_PER_APP != 1200:
    raise ValueError("Target reviews per app must remain 1,200.")

if set(app_config_df["source"].dropna().unique()) != {SOURCE}:
    raise ValueError("Source in phase2_apps does not match expected Google Play source.")

print("Confirmed same 10 apps from phase2_apps.")
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)

,app_name,app_id,source,language,country,title_from_store,score_from_store,ratings_from_store,installs_from_store
0,YouTube,com.google.android.youtube,google_play,en,us,YouTube,3.861226,170926411,"10,000,000,000+"
1,TikTok,com.zhiliaoapp.musically,google_play,en,us,"TikTok - Videos, Shop & LIVE",3.991081,69280732,"1,000,000,000+"
2,Spotify,com.spotify.music,google_play,en,us,Spotify: Music and Podcasts,4.335805,35890307,"1,000,000,000+"
3,Instagram,com.instagram.android,google_play,en,us,Instagram,4.002019,168329152,"5,000,000,000+"
4,Uber,com.ubercab,google_play,en,us,Uber - Request a ride,4.743543,19073493,"1,000,000,000+"
5,DoorDash,com.dd.doordash,google_play,en,us,"DoorDash: Food, Grocery, More",4.657277,6029493,"50,000,000+"
6,Duolingo,com.duolingo,google_play,en,us,Duolingo: Language Lessons,4.726991,47254150,"500,000,000+"
7,Google Maps,com.google.android.apps.maps,google_play,en,us,Google Maps,3.248390,19469101,"10,000,000,000+"
8,Netflix,com.netflix.mediaclient,google_play,en,us,Netflix,3.870859,15170396,"1,000,000,000+"
9,Reddit,com.reddit.frontpage,google_play,en,us,Reddit,4.585813,4691886,"100,000,000+"


Confirmed same 10 apps from phase2_apps.
Target reviews per app: 1200


## 6. Hard validation before cadence collection

This prevents Run A from using the wrong database, wrong schema, wrong target, or wrong app list.

In [8]:
required_tables = [
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
]

existing_tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
""", conn)["name"].tolist()

missing_tables = [t for t in required_tables if t not in existing_tables]

if missing_tables:
    raise ValueError(f"Missing required Phase 2 tables: {missing_tables}")

if raw_rows_before < 17000:
    raise ValueError(
        f"Raw review table only has {raw_rows_before} rows. "
        "This does not look like the fixed Day 3 database."
    )

if cleaned_rows_before < 17000:
    raise ValueError(
        f"Cleaned review table only has {cleaned_rows_before} rows. "
        "This does not look like the fixed Day 3 database."
    )

if prior_run_count < 3:
    raise ValueError("Expected at least 3 prior Phase 2 runs before cadence testing.")

if len(app_config_df) != 10:
    raise ValueError("Expected 10 apps for cadence test.")

if TARGET_REVIEWS_PER_APP != 1200:
    raise ValueError("Expected 1,200 target reviews per app.")

print("Hard validation passed.")
print("This is a valid cadence test continuation from the fixed Day 3 database.")

Hard validation passed.
This is a valid cadence test continuation from the fixed Day 3 database.


## 7. Helper functions

These functions match the Phase 2 schema and keep the same duplicate-prevention logic.

In [9]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def to_iso_utc(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, datetime):
        dt = value
    else:
        try:
            dt = pd.to_datetime(value).to_pydatetime()
        except Exception:
            return str(value)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc).isoformat(timespec="seconds")

def make_hash(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def make_review_key(source, app_id, review_id):
    return make_hash(f"{source}|{app_id}|{review_id}")

def make_flag_id(run_id, review_key, flag_name):
    return make_hash(f"{run_id}|{review_key}|{flag_name}")

def clean_content(text):
    if text is None:
        return None

    cleaned = str(text).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned

def json_ready(value):
    if isinstance(value, datetime):
        return to_iso_utc(value)
    return value

def raw_review_to_json(raw_review):
    cleaned = {}

    for k, v in raw_review.items():
        cleaned[k] = json_ready(v)

    return json.dumps(cleaned, ensure_ascii=False)

def normalize_review(raw_review, app_id, app_name, fetched_at, run_id):
    review_id = raw_review.get("reviewId")

    review_key = None
    if review_id:
        review_key = make_review_key(SOURCE, app_id, review_id)

    content_raw = raw_review.get("content")
    reply_content_raw = raw_review.get("replyContent")
    app_version = raw_review.get("reviewCreatedVersion") or raw_review.get("appVersion")

    row = {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": raw_review.get("userName"),
        "user_image": raw_review.get("userImage"),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get("thumbsUpCount"),
        "review_created_at": to_iso_utc(raw_review.get("at")),
        "reply_content_raw": reply_content_raw,
        "replied_at": to_iso_utc(raw_review.get("repliedAt")),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": run_id,
        "raw_json": raw_review_to_json(raw_review),
    }

    return row

def make_cleaned_row(raw_row, cleaned_at):
    content_cleaned = clean_content(raw_row.get("content_raw"))

    return {
        "review_key": raw_row.get("review_key"),
        "source": raw_row.get("source"),
        "app_id": raw_row.get("app_id"),
        "content_cleaned": content_cleaned,
        "content_length": len(content_cleaned) if content_cleaned is not None else None,
        "has_developer_reply": 1 if raw_row.get("reply_content_raw") not in [None, ""] else 0,
        "score": raw_row.get("score"),
        "review_created_at": raw_row.get("review_created_at"),
        "app_version": raw_row.get("app_version"),
        "cleaned_at": cleaned_at,
        "run_id": raw_row.get("run_id"),
    }

def generate_quality_flags(raw_row, run_id):
    flags = []
    created_at = utc_now_iso()

    review_key = raw_row.get("review_key")
    app_id = raw_row.get("app_id")

    if review_key is None:
        return flags

    def add_flag(flag_name, severity, flag_value):
        flags.append({
            "flag_id": make_flag_id(run_id, review_key, flag_name),
            "review_key": review_key,
            "run_id": run_id,
            "app_id": app_id,
            "flag_name": flag_name,
            "flag_severity": severity,
            "flag_value": flag_value,
            "created_at": created_at,
        })

    if raw_row.get("content_raw") is None:
        add_flag("missing_content", "warning", "missing")
    elif str(raw_row.get("content_raw")).strip() == "":
        add_flag("empty_content", "warning", "empty")

    if raw_row.get("score") is None:
        add_flag("missing_score", "warning", "missing")
    elif raw_row.get("score") not in [1, 2, 3, 4, 5]:
        add_flag("invalid_score", "warning", str(raw_row.get("score")))

    if raw_row.get("review_created_at") is None:
        add_flag("missing_review_date", "warning", "missing")

    if raw_row.get("app_version") in [None, ""]:
        add_flag("missing_app_version", "info", "missing")

    if raw_row.get("reply_content_raw") in [None, ""]:
        add_flag("missing_developer_reply", "info", "missing")

    return flags

def insert_raw_review(conn, raw_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "app_name",
        "review_id",
        "user_name",
        "user_image",
        "content_raw",
        "score",
        "thumbs_up_count",
        "review_created_at",
        "reply_content_raw",
        "replied_at",
        "app_version",
        "fetched_at",
        "run_id",
        "raw_json",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_raw
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [raw_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return int(cur.rowcount)

def insert_cleaned_review(conn, cleaned_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "content_cleaned",
        "content_length",
        "has_developer_reply",
        "score",
        "review_created_at",
        "app_version",
        "cleaned_at",
        "run_id",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_cleaned
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [cleaned_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return int(cur.rowcount)

def insert_quality_flag(conn, flag_row):
    columns = [
        "flag_id",
        "review_key",
        "run_id",
        "app_id",
        "flag_name",
        "flag_severity",
        "flag_value",
        "created_at",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_quality_flags
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [flag_row.get(col) for col in columns]
    cur = conn.cursor()
    cur.execute(sql, values)

    return int(cur.rowcount)

## 8. Create the cadence Run A record

This run is marked as a twice-daily same-day second collection test.

In [10]:
run_started_at = utc_now_iso()
apps_included = ", ".join(app_config_df["app_name"].tolist())

cur.execute("""
    INSERT INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", (
    str(RUN_ID),
    str(RUN_LABEL),
    "phase2",
    str(FREQUENCY_LABEL),
    str(SOURCE),
    str(LANGUAGE),
    str(COUNTRY),
    int(TARGET_REVIEWS_PER_APP),
    int(len(app_config_df)),
    str(apps_included),
    str(run_started_at),
    "running",
    int(0),
    int(0),
    int(0),
    int(0),
    int(0),
    int(0),
    float(db_size_before_mb),
    int(raw_rows_before),
    "Phase 2 cadence Run A. Same 10 apps and same 1,200-review target. Tests twice-daily same-day second collection against the Day 1-Day 3 once-daily baseline."
))

conn.commit()

print("Cadence Run A record created.")
print("Run started at:", run_started_at)

Cadence Run A record created.
Run started at: 2026-07-09T21:14:19.095514+00:00


## 9. Run cadence collection

This collects 1,200 newest reviews per app again using the same setup.

Because this is a higher-frequency test, the important metrics are:

- duplicate rate
- new insert rate
- app-level new review capture
- runtime
- database growth
- quality flags
- errors

In [11]:
collection_start_time = time.time()

app_summaries = []
quality_flags_created = []
new_review_rows = []
errors = []

print("Starting Phase 2 Cadence Test Run A...")
print("Run ID:", RUN_ID)

for i, app_row in app_config_df.iterrows():
    app_start_time = time.time()

    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    print("\n" + "=" * 90)
    print(f"[{i + 1}/{len(app_config_df)}] Collecting {app_name} ({app_id})")

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    quality_flag_count = 0
    quality_flags_inserted = 0
    error_message = ""

    missing_review_id_count = 0
    missing_content_count = 0
    empty_content_count = 0
    missing_score_count = 0
    invalid_score_count = 0
    missing_review_date_count = 0
    missing_app_version_count = 0
    missing_developer_reply_count = 0

    min_review_date = None
    max_review_date = None

    try:
        fetched_at = utc_now_iso()

        fetched_reviews, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=SORT_METHOD,
            count=TARGET_REVIEWS_PER_APP
        )

        records_fetched = int(len(fetched_reviews))

        normalized_rows = [
            normalize_review(raw_review, app_id, app_name, fetched_at, RUN_ID)
            for raw_review in fetched_reviews
        ]

        valid_review_keys = [
            row["review_key"]
            for row in normalized_rows
            if row.get("review_key") is not None
        ]

        unique_reviews_in_batch = int(len(set(valid_review_keys)))
        duplicate_reviews_in_batch = int(len(valid_review_keys) - unique_reviews_in_batch)

        review_dates = [
            row["review_created_at"]
            for row in normalized_rows
            if row.get("review_created_at") is not None
        ]

        if review_dates:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)

        for raw_row in normalized_rows:
            if raw_row.get("review_id") in [None, ""]:
                missing_review_id_count += 1
                continue

            if raw_row.get("content_raw") is None:
                missing_content_count += 1
            elif str(raw_row.get("content_raw")).strip() == "":
                empty_content_count += 1

            if raw_row.get("score") is None:
                missing_score_count += 1
            elif raw_row.get("score") not in [1, 2, 3, 4, 5]:
                invalid_score_count += 1

            if raw_row.get("review_created_at") is None:
                missing_review_date_count += 1

            if raw_row.get("app_version") in [None, ""]:
                missing_app_version_count += 1

            if raw_row.get("reply_content_raw") in [None, ""]:
                missing_developer_reply_count += 1

            inserted_raw = insert_raw_review(conn, raw_row)

            if inserted_raw == 1:
                cleaned_row = make_cleaned_row(raw_row, utc_now_iso())
                insert_cleaned_review(conn, cleaned_row)

                new_records_inserted += 1
                new_review_rows.append(raw_row)
            else:
                duplicates_skipped += 1

            flags = generate_quality_flags(raw_row, RUN_ID)
            quality_flag_count += len(flags)

            for flag in flags:
                quality_flags_inserted += insert_quality_flag(conn, flag)
                quality_flags_created.append(flag)

        conn.commit()

    except Exception as e:
        error_message = repr(e)
        errors.append({"app_name": app_name, "app_id": app_id, "error_message": error_message})
        print("ERROR:", error_message)

    app_runtime_seconds = float(time.time() - app_start_time)

    app_summary = {
        "run_id": RUN_ID,
        "app_name": app_name,
        "app_id": app_id,
        "target_reviews": int(TARGET_REVIEWS_PER_APP),
        "records_fetched": int(records_fetched),
        "unique_reviews_in_batch": int(unique_reviews_in_batch),
        "duplicate_reviews_in_batch": int(duplicate_reviews_in_batch),
        "new_records_inserted": int(new_records_inserted),
        "duplicates_skipped": int(duplicates_skipped),
        "runtime_seconds": app_runtime_seconds,
        "min_review_date": min_review_date,
        "max_review_date": max_review_date,
        "missing_review_id_count": int(missing_review_id_count),
        "missing_content_count": int(missing_content_count),
        "empty_content_count": int(empty_content_count),
        "missing_score_count": int(missing_score_count),
        "invalid_score_count": int(invalid_score_count),
        "missing_review_date_count": int(missing_review_date_count),
        "missing_app_version_count": int(missing_app_version_count),
        "missing_developer_reply_count": int(missing_developer_reply_count),
        "quality_flag_count": int(quality_flag_count),
        "error_message": error_message,
    }

    app_summaries.append(app_summary)

    cur.execute("""
        INSERT OR REPLACE INTO phase2_app_run_summary (
            run_id,
            app_name,
            app_id,
            target_reviews,
            records_fetched,
            unique_reviews_in_batch,
            duplicate_reviews_in_batch,
            new_records_inserted,
            duplicates_skipped,
            runtime_seconds,
            min_review_date,
            max_review_date,
            missing_review_id_count,
            missing_content_count,
            empty_content_count,
            missing_score_count,
            invalid_score_count,
            missing_review_date_count,
            missing_app_version_count,
            missing_developer_reply_count,
            quality_flag_count,
            error_message
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        str(app_summary["run_id"]),
        str(app_summary["app_name"]),
        str(app_summary["app_id"]),
        int(app_summary["target_reviews"]),
        int(app_summary["records_fetched"]),
        int(app_summary["unique_reviews_in_batch"]),
        int(app_summary["duplicate_reviews_in_batch"]),
        int(app_summary["new_records_inserted"]),
        int(app_summary["duplicates_skipped"]),
        float(app_summary["runtime_seconds"]),
        app_summary["min_review_date"],
        app_summary["max_review_date"],
        int(app_summary["missing_review_id_count"]),
        int(app_summary["missing_content_count"]),
        int(app_summary["empty_content_count"]),
        int(app_summary["missing_score_count"]),
        int(app_summary["invalid_score_count"]),
        int(app_summary["missing_review_date_count"]),
        int(app_summary["missing_app_version_count"]),
        int(app_summary["missing_developer_reply_count"]),
        int(app_summary["quality_flag_count"]),
        str(app_summary["error_message"]),
    ))

    conn.commit()

    print(
        f"Fetched={records_fetched:,} | "
        f"New inserts={new_records_inserted:,} | "
        f"Duplicates skipped={duplicates_skipped:,} | "
        f"Runtime={app_runtime_seconds:.2f}s | "
        f"Quality flags={quality_flag_count:,}"
    )

    time.sleep(REQUEST_SLEEP_SECONDS)

collection_runtime_seconds = float(time.time() - collection_start_time)

print("\n" + "=" * 90)
print("Cadence Test Run A finished.")
print(f"Runtime: {collection_runtime_seconds:.2f} seconds")

Starting Phase 2 Cadence Test Run A...
Run ID: phase2_cadence_runA_twice_daily_test_20260709_210858

[1/10] Collecting YouTube (com.google.android.youtube)
Fetched=1,200 | New inserts=1,199 | Duplicates skipped=1 | Runtime=1.22s | Quality flags=1,236

[2/10] Collecting TikTok (com.zhiliaoapp.musically)
Fetched=1,200 | New inserts=622 | Duplicates skipped=578 | Runtime=0.68s | Quality flags=649

[3/10] Collecting Spotify (com.spotify.music)
Fetched=1,200 | New inserts=580 | Duplicates skipped=620 | Runtime=0.74s | Quality flags=1,280

[4/10] Collecting Instagram (com.instagram.android)
Fetched=1,200 | New inserts=1,198 | Duplicates skipped=2 | Runtime=1.02s | Quality flags=1,590

[5/10] Collecting Uber (com.ubercab)
Fetched=1,200 | New inserts=317 | Duplicates skipped=883 | Runtime=0.98s | Quality flags=1,368

[6/10] Collecting DoorDash (com.dd.doordash)
Fetched=1,200 | New inserts=73 | Duplicates skipped=1,127 | Runtime=0.57s | Quality flags=1,333

[7/10] Collecting Duolingo (com.duoli

## 10. Save Run A app-level summary

In [12]:
app_summary_df = pd.DataFrame(app_summaries)

app_summary_path = OUTPUT_DIR / "phase2_cadence_runA_app_level_summary.csv"
app_summary_df.to_csv(app_summary_path, index=False)

print("Saved:", app_summary_path)

display(app_summary_df)

Saved: /content/outputs/phase2_cadence_runA_app_level_summary.csv


,run_id,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,runtime_seconds,...,missing_review_id_count,missing_content_count,empty_content_count,missing_score_count,invalid_score_count,missing_review_date_count,missing_app_version_count,missing_developer_reply_count,quality_flag_count,error_message
0,phase2_cadence_runA_twice_daily_test_20260709_...,YouTube,com.google.android.youtube,1200,1200,1200,0,1199,1,1.218204,...,0,0,0,0,0,0,36,1200,1236,
1,phase2_cadence_runA_twice_daily_test_20260709_...,TikTok,com.zhiliaoapp.musically,1200,1200,1200,0,622,578,0.683194,...,0,0,0,0,0,0,462,187,649,
2,phase2_cadence_runA_twice_daily_test_20260709_...,Spotify,com.spotify.music,1200,1200,1200,0,580,620,0.741731,...,0,0,0,0,0,0,205,1075,1280,
3,phase2_cadence_runA_twice_daily_test_20260709_...,Instagram,com.instagram.android,1200,1200,1200,0,1198,2,1.020985,...,0,0,0,0,0,0,390,1200,1590,
4,phase2_cadence_runA_twice_daily_test_20260709_...,Uber,com.ubercab,1200,1200,1200,0,317,883,0.980962,...,0,0,0,0,0,0,170,1198,1368,
5,phase2_cadence_runA_twice_daily_test_20260709_...,DoorDash,com.dd.doordash,1200,1200,1200,0,73,1127,0.570115,...,0,0,0,0,0,0,133,1200,1333,
6,phase2_cadence_runA_twice_daily_test_20260709_...,Duolingo,com.duolingo,1200,1200,1200,0,6,1194,0.777184,...,0,0,0,0,0,0,94,1200,1294,
7,phase2_cadence_runA_twice_daily_test_20260709_...,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,184,1016,0.652907,...,0,0,0,0,0,0,28,936,964,
8,phase2_cadence_runA_twice_daily_test_20260709_...,Netflix,com.netflix.mediaclient,1200,1200,1200,0,123,1077,0.715119,...,0,0,0,0,0,0,355,1200,1555,
9,phase2_cadence_runA_twice_daily_test_20260709_...,Reddit,com.reddit.frontpage,1200,1200,1200,0,93,1107,0.680379,...,0,0,0,0,0,0,252,1200,1452,


## 11. Update Run A run-level summary

In [13]:
run_finished_at = utc_now_iso()

raw_rows_after = int(pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
    conn
)["n"].iloc[0])

cleaned_rows_after = int(pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
    conn
)["n"].iloc[0])

db_size_after_mb = float(DB_PATH.stat().st_size / (1024 * 1024))

records_fetched_total = int(app_summary_df["records_fetched"].sum())
new_records_inserted_total = int(app_summary_df["new_records_inserted"].sum())
duplicates_skipped_total = int(app_summary_df["duplicates_skipped"].sum())
errors_total = int((app_summary_df["error_message"].fillna("") != "").sum())
quality_flag_total = int(app_summary_df["quality_flag_count"].sum())
quality_flags_inserted_total = int(len(quality_flags_created))

apps_failed = ", ".join(app_summary_df.loc[
    app_summary_df["error_message"].fillna("") != "",
    "app_name"
].tolist())

review_rows_growth = int(raw_rows_after - raw_rows_before)
db_size_growth_mb = float(db_size_after_mb - db_size_before_mb)

status = "completed" if errors_total == 0 else "completed_with_errors"

cur.execute("""
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        apps_failed = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?
""", (
    str(run_finished_at),
    float(collection_runtime_seconds),
    str(status),
    int(records_fetched_total),
    int(new_records_inserted_total),
    int(duplicates_skipped_total),
    int(errors_total),
    str(apps_failed),
    int(quality_flag_total),
    int(quality_flags_inserted_total),
    float(db_size_after_mb),
    float(db_size_growth_mb),
    int(raw_rows_after),
    int(review_rows_growth),
    str(RUN_ID),
))

conn.commit()

run_summary_df = pd.read_sql_query("""
    SELECT
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        apps_failed,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        notes
    FROM phase2_ingestion_runs
    WHERE run_id = ?
""", conn, params=(RUN_ID,))

run_summary_path = OUTPUT_DIR / "phase2_cadence_runA_run_summary.csv"
run_summary_df.to_csv(run_summary_path, index=False)

print("Saved:", run_summary_path)

display(run_summary_df)

Saved: /content/outputs/phase2_cadence_runA_run_summary.csv


,run_id,run_label,phase,frequency_label,source,language,country,target_reviews_per_app,app_count,apps_included,...,apps_failed,quality_flag_total,quality_flags_inserted,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth,notes
0,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,phase2,twice_daily_same_day_second_run,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, Doo...",...,,12721,12721,43.761719,55.523438,11.761719,17815,22210,4395,Phase 2 cadence Run A. Same 10 apps and same 1...


## 12. Print compact Run A summary

In [14]:
duplicate_rate = (
    duplicates_skipped_total / records_fetched_total
    if records_fetched_total > 0
    else None
)

new_insert_rate = (
    new_records_inserted_total / records_fetched_total
    if records_fetched_total > 0
    else None
)

print("PHASE 2 CADENCE TEST RUN A SUMMARY")
print("-" * 70)
print(f"Total fetched records:      {records_fetched_total:,}")
print(f"New inserts:                {new_records_inserted_total:,}")
print(f"Duplicates skipped:         {duplicates_skipped_total:,}")
print(f"Duplicate rate:             {duplicate_rate:.2%}" if duplicate_rate is not None else "Duplicate rate:             N/A")
print(f"New insert rate:            {new_insert_rate:.2%}" if new_insert_rate is not None else "New insert rate:            N/A")
print(f"Runtime seconds:            {collection_runtime_seconds:.2f}")
print(f"Runtime minutes:            {collection_runtime_seconds / 60:.2f}")
print(f"Raw rows before:            {raw_rows_before:,}")
print(f"Raw rows after:             {raw_rows_after:,}")
print(f"Raw row growth:             {review_rows_growth:,}")
print(f"Cleaned rows before:        {cleaned_rows_before:,}")
print(f"Cleaned rows after:         {cleaned_rows_after:,}")
print(f"Database size before:       {db_size_before_mb:.2f} MB")
print(f"Database size after:        {db_size_after_mb:.2f} MB")
print(f"Database growth:            {db_size_growth_mb:.2f} MB")
print(f"Errors:                     {errors_total}")
print(f"Quality flags:              {quality_flag_total:,}")

PHASE 2 CADENCE TEST RUN A SUMMARY
----------------------------------------------------------------------
Total fetched records:      12,000
New inserts:                4,395
Duplicates skipped:         7,605
Duplicate rate:             63.38%
New insert rate:            36.62%
Runtime seconds:            28.14
Runtime minutes:            0.47
Raw rows before:            17,815
Raw rows after:             22,210
Raw row growth:             4,395
Cleaned rows before:        17,815
Cleaned rows after:         22,210
Database size before:       43.76 MB
Database size after:        55.52 MB
Database growth:            11.76 MB
Errors:                     0
Quality flags:              12,721


## 13. Save Run A quality flags

In [15]:
quality_flags_df = pd.DataFrame(quality_flags_created)

quality_flags_path = OUTPUT_DIR / "phase2_cadence_runA_quality_flags.csv"
quality_flags_df.to_csv(quality_flags_path, index=False)

print("Saved:", quality_flags_path)
print("Quality flag rows:", len(quality_flags_df))

if len(quality_flags_df) > 0:
    display(quality_flags_df.head(20))
else:
    display(pd.DataFrame({"message": ["No quality flags created."]}))

Saved: /content/outputs/phase2_cadence_runA_quality_flags.csv
Quality flag rows: 12721


,flag_id,review_key,run_id,app_id,flag_name,flag_severity,flag_value,created_at
0,66b6b4c9b9857c575a0e938193be5a02e411fdef9c3ffa...,860ef7110cae18b4a12f672930dccb6d0c0588ab392d32...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.096199+00:00
1,4534a6a1b9bb7e52ac2878e47af583032fbacd34560eb8...,38c42071efe09eda8796e90e65f9a30555e7cf0a321d89...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.096609+00:00
2,6a66f5777c4e66a22a750cd402fd42b19c9190bf9387a4...,0ab6c1274ddc7c4da3f1e3b84af9bcd015ae7f011b0049...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.097018+00:00
3,15c8f885c80fdc85a9cf3d4d038bff5e6dc331fd574339...,bd45f74ee8c8567f8fb4cf78ecdd155b0a3f3362ba6b6f...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.097272+00:00
4,ff07ddea34256d9e5281383a9ba89d021ca8e1dbb1a3a8...,2866b24cba3850a6eb93f1ab509d8c89805a29576648f3...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.097555+00:00
5,d95fea23430f23679e8dc459521d1d07baecccb728667a...,e149f460176402615009bcb4b11660d31c33444b94e61b...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.097787+00:00
6,23bbae24064e4bfe9892767da4c42fa42a1a1edcf2fdda...,2aa6c5d97b8d11aa517964322487549176ee168fa6abd7...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.098041+00:00
7,17cf1124fe0cf133f2bb8f230a8345ac865fe5c747518f...,8d99e8ae02b2760b902ba20e57d118362cf0cb7dbdd93e...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.098346+00:00
8,8ba9ec06f663dd6ecfa179eb2609112c236197a5a1917f...,2c13febdbd6c7808a6e354677c5146e6ea1b587c11ba24...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.098526+00:00
9,4711fb80c13337652cd87623f8b33b25d5c26490597f42...,f36b4bf9d2cd1392269ede62b22cbe95ed728d94159ad0...,phase2_cadence_runA_twice_daily_test_20260709_...,com.google.android.youtube,missing_developer_reply,info,missing,2026-07-09T21:14:36.098695+00:00


## 14. Compare once-daily baseline vs. twice-daily Run A

This directly addresses the operating assumption test requested by John.

In [16]:
cadence_comparison_df = pd.read_sql_query("""
    SELECT
        run_label,
        run_id,
        frequency_label,
        run_started_at,
        run_finished_at,
        target_reviews_per_app,
        app_count,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(duplicates_skipped_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS duplicate_rate,
        CASE
            WHEN records_fetched_total > 0
            THEN ROUND(CAST(new_records_inserted_total AS REAL) / records_fetched_total, 4)
            ELSE NULL
        END AS new_insert_rate,
        runtime_seconds,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total,
        quality_flag_total,
        status
    FROM phase2_ingestion_runs
    WHERE
        run_label LIKE 'phase2_day%'
        OR run_label = ?
    ORDER BY run_started_at
""", conn, params=(RUN_LABEL,))

cadence_comparison_path = OUTPUT_DIR / "phase2_cadence_comparison_through_runA.csv"
cadence_comparison_df.to_csv(cadence_comparison_path, index=False)

print("Saved:", cadence_comparison_path)

display(cadence_comparison_df)

Saved: /content/outputs/phase2_cadence_comparison_through_runA.csv


,run_label,run_id,frequency_label,run_started_at,run_finished_at,target_reviews_per_app,app_count,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,...,runtime_seconds,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb,errors_total,quality_flag_total,status
0,phase2_day1_controlled_scale,phase2_day1_controlled_scale_20260708_034445,once_daily_baseline,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,1200,10,12000,12000,0,...,23.271117,0,12000,12000,1.832031,25.492188,23.660156,0,12633,completed
1,phase2_day2_daily_followup,phase2_day2_daily_followup_20260708_041135,daily_followup,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,1200,10,12000,156,11844,...,75.799585,12000,12156,156,25.492188,30.187500,4.695312,0,12638,completed
2,phase2_day3_controlled_repeated_run,phase2_day3_controlled_repeated_run_20260709_0...,daily_followup_controlled_baseline,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,1200,10,12000,5659,6341,...,29.269241,12156,17815,5659,30.187500,43.761719,13.574219,0,12696,completed
3,phase2_cadence_runA_twice_daily_test,phase2_cadence_runA_twice_daily_test_20260709_...,twice_daily_same_day_second_run,2026-07-09T21:14:19.095514+00:00,2026-07-09T21:15:07.605058+00:00,1200,10,12000,4395,7605,...,28.142863,17815,22210,4395,43.761719,55.523438,11.761719,0,12721,completed


## 15. Compare app-level behavior across baseline and Run A

This helps identify high-activity apps, low-activity apps, duplicate-heavy apps, and apps with unusual quality flag patterns.

In [17]:
cadence_app_comparison_df = pd.read_sql_query("""
    SELECT
        s.run_id,
        r.run_label,
        r.frequency_label,
        r.run_started_at,
        s.app_name,
        s.app_id,
        s.target_reviews,
        s.records_fetched,
        s.unique_reviews_in_batch,
        s.duplicate_reviews_in_batch,
        s.new_records_inserted,
        s.duplicates_skipped,
        CASE
            WHEN s.records_fetched > 0
            THEN ROUND(CAST(s.duplicates_skipped AS REAL) / s.records_fetched, 4)
            ELSE NULL
        END AS duplicate_rate,
        CASE
            WHEN s.records_fetched > 0
            THEN ROUND(CAST(s.new_records_inserted AS REAL) / s.records_fetched, 4)
            ELSE NULL
        END AS new_insert_rate,
        s.runtime_seconds,
        s.min_review_date,
        s.max_review_date,
        s.quality_flag_count,
        s.error_message
    FROM phase2_app_run_summary s
    LEFT JOIN phase2_ingestion_runs r
        ON s.run_id = r.run_id
    WHERE
        r.run_label LIKE 'phase2_day%'
        OR r.run_label = ?
    ORDER BY s.app_name, r.run_started_at
""", conn, params=(RUN_LABEL,))

cadence_app_comparison_path = OUTPUT_DIR / "phase2_cadence_app_comparison_through_runA.csv"
cadence_app_comparison_df.to_csv(cadence_app_comparison_path, index=False)

print("Saved:", cadence_app_comparison_path)

display(cadence_app_comparison_df)

Saved: /content/outputs/phase2_cadence_app_comparison_through_runA.csv


,run_id,run_label,frequency_label,run_started_at,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,duplicate_rate,new_insert_rate,runtime_seconds,min_review_date,max_review_date,quality_flag_count,error_message
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,2026-07-08T03:44:46.106977+00:00,DoorDash,com.dd.doordash,1200,1200,1200,0,1200,0,0.0000,1.0000,0.830000,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,1326,
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,2026-07-08T04:16:09.351152+00:00,DoorDash,com.dd.doordash,1200,1200,1200,0,0,1200,1.0000,0.0000,0.730000,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,1326,
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,2026-07-09T04:04:39.642719+00:00,DoorDash,com.dd.doordash,1200,1200,1200,0,130,1070,0.8917,0.1083,0.711403,2026-06-29T20:05:31+00:00,2026-07-08T03:59:09+00:00,1326,
3,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,2026-07-09T21:14:19.095514+00:00,DoorDash,com.dd.doordash,1200,1200,1200,0,73,1127,0.9392,0.0608,0.570115,2026-06-30T03:00:37+00:00,2026-07-08T21:03:08+00:00,1333,
4,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,2026-07-08T03:44:46.106977+00:00,Duolingo,com.duolingo,1200,1200,1200,0,1200,0,0.0000,1.0000,1.370000,2026-07-06T04:53:03+00:00,2026-07-07T03:43:18+00:00,1277,
5,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,2026-07-08T04:16:09.351152+00:00,Duolingo,com.duolingo,1200,1200,1200,0,28,1172,0.9767,0.0233,0.760000,2026-07-06T05:42:10+00:00,2026-07-07T04:15:00+00:00,1276,
6,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,2026-07-09T04:04:39.642719+00:00,Duolingo,com.duolingo,1200,1200,1200,0,856,344,0.2867,0.7133,1.305671,2026-07-06T18:13:17+00:00,2026-07-08T02:31:26+00:00,1294,
7,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,2026-07-09T21:14:19.095514+00:00,Duolingo,com.duolingo,1200,1200,1200,0,6,1194,0.9950,0.0050,0.777184,2026-07-06T18:18:00+00:00,2026-07-08T15:57:17+00:00,1294,
8,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,2026-07-08T03:44:46.106977+00:00,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,1200,0,0.0000,1.0000,0.850000,2026-06-30T02:30:54+00:00,2026-07-07T03:30:28+00:00,965,
9,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,2026-07-08T04:16:09.351152+00:00,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,3,1197,0.9975,0.0025,0.870000,2026-06-30T03:12:23+00:00,2026-07-07T04:05:23+00:00,965,


## 16. Rank Run A apps by new inserted records

This checks whether higher-activity apps are producing meaningfully more new reviews.

In [18]:
runA_app_ranking_df = app_summary_df.copy()

runA_app_ranking_df["duplicate_rate"] = runA_app_ranking_df.apply(
    lambda row: row["duplicates_skipped"] / row["records_fetched"]
    if row["records_fetched"] else None,
    axis=1
)

runA_app_ranking_df["new_insert_rate"] = runA_app_ranking_df.apply(
    lambda row: row["new_records_inserted"] / row["records_fetched"]
    if row["records_fetched"] else None,
    axis=1
)

runA_app_ranking_df = runA_app_ranking_df.sort_values(
    by=["new_records_inserted", "records_fetched"],
    ascending=[False, False]
).reset_index(drop=True)

runA_app_ranking_path = OUTPUT_DIR / "phase2_cadence_runA_app_new_insert_ranking.csv"
runA_app_ranking_df.to_csv(runA_app_ranking_path, index=False)

print("Saved:", runA_app_ranking_path)

display(runA_app_ranking_df[[
    "app_name",
    "app_id",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "new_insert_rate",
    "runtime_seconds",
    "quality_flag_count",
    "min_review_date",
    "max_review_date",
    "error_message"
]])

Saved: /content/outputs/phase2_cadence_runA_app_new_insert_ranking.csv


,app_name,app_id,records_fetched,new_records_inserted,duplicates_skipped,duplicate_rate,new_insert_rate,runtime_seconds,quality_flag_count,min_review_date,max_review_date,error_message
0,YouTube,com.google.android.youtube,1200,1199,1,0.000833,0.999167,1.218204,1236,2026-07-08T10:17:11+00:00,2026-07-08T21:13:22+00:00,
1,Instagram,com.instagram.android,1200,1198,2,0.001667,0.998333,1.020985,1590,2026-07-08T11:26:23+00:00,2026-07-08T21:12:32+00:00,
2,TikTok,com.zhiliaoapp.musically,1200,622,578,0.481667,0.518333,0.683194,649,2026-07-07T07:19:54+00:00,2026-07-08T21:14:15+00:00,
3,Spotify,com.spotify.music,1200,580,620,0.516667,0.483333,0.741731,1280,2026-07-07T08:53:59+00:00,2026-07-08T21:14:19+00:00,
4,Uber,com.ubercab,1200,317,883,0.735833,0.264167,0.980962,1368,2026-07-05T21:38:18+00:00,2026-07-08T21:12:10+00:00,
5,Google Maps,com.google.android.apps.maps,1200,184,1016,0.846667,0.153333,0.652907,964,2026-07-02T08:11:36+00:00,2026-07-08T21:14:19+00:00,
6,Netflix,com.netflix.mediaclient,1200,123,1077,0.897500,0.102500,0.715119,1555,2026-06-29T08:10:20+00:00,2026-07-08T21:01:32+00:00,
7,Reddit,com.reddit.frontpage,1200,93,1107,0.922500,0.077500,0.680379,1452,2026-06-29T01:46:57+00:00,2026-07-08T20:43:50+00:00,
8,DoorDash,com.dd.doordash,1200,73,1127,0.939167,0.060833,0.570115,1333,2026-06-30T03:00:37+00:00,2026-07-08T21:03:08+00:00,
9,Duolingo,com.duolingo,1200,6,1194,0.995000,0.005000,0.777184,1294,2026-07-06T18:18:00+00:00,2026-07-08T15:57:17+00:00,


## 17. Save sample newly inserted reviews

This keeps the GitHub output lightweight.

In [19]:
new_reviews_df = pd.DataFrame(new_review_rows)

sample_size = min(200, len(new_reviews_df))

if sample_size > 0:
    sample_new_reviews_df = new_reviews_df.sample(sample_size, random_state=42)
else:
    sample_new_reviews_df = new_reviews_df

sample_new_reviews_path = OUTPUT_DIR / "phase2_cadence_runA_sample_new_reviews.csv"
sample_new_reviews_df.to_csv(sample_new_reviews_path, index=False)

print("New Run A reviews inserted:", len(new_reviews_df))
print("Saved:", sample_new_reviews_path)

display(sample_new_reviews_df.head(20))

New Run A reviews inserted: 4395
Saved: /content/outputs/phase2_cadence_runA_sample_new_reviews.csv


,review_key,source,app_id,app_name,review_id,user_name,user_image,content_raw,score,thumbs_up_count,review_created_at,reply_content_raw,replied_at,app_version,fetched_at,run_id,raw_json
3942,8ba7a7a1b75a6c99aff81ca27f957f5669a4dd7f583fe7...,google_play,com.dd.doordash,DoorDash,05d0e537-aa45-4aae-9980-af85848d4899,Ashley W,https://play-lh.googleusercontent.com/a-/ALV-U...,I'll put review after I get my food,5,0,2026-07-08T17:26:01+00:00,None,None,15.280.2,2026-07-09T21:14:49.802383+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""05d0e537-aa45-4aae-9980-af85848d..."
2658,349f1a1a3ab58d11a6b23fa80ee6181220908c955befc7...,google_play,com.instagram.android,Instagram,24a3a7e5-d0d6-4517-b52c-4b22f1f975b6,Ayan Qureshi,https://play-lh.googleusercontent.com/a/ACg8oc...,Nice app,5,0,2026-07-08T17:40:57+00:00,None,None,None,2026-07-09T21:14:43.782074+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""24a3a7e5-d0d6-4517-b52c-4b22f1f9..."
1532,aa9dcc0e026a539570d423578bae5fe6345f0fbe91c9d4...,google_play,com.zhiliaoapp.musically,TikTok,bcda207a-f5d6-4953-811f-b7ad00358a04,Peninah Agwang,https://play-lh.googleusercontent.com/a/ACg8oc...,Its delaying to open,2,0,2026-07-08T12:03:38+00:00,"Hi, sorry for the inconvenience. To help us lo...",2026-07-08T12:20:12+00:00,45.7.3,2026-07-09T21:14:38.342700+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""bcda207a-f5d6-4953-811f-b7ad0035..."
414,9f2fc91ed8969688e9d14b953225b27115fcb79974dc1f...,google_play,com.google.android.youtube,YouTube,522c9be4-ab2a-48e1-98ba-d0b7e5168914,Sima Khairnar,https://play-lh.googleusercontent.com/a/ACg8oc...,Bhushan narkar,5,1,2026-07-08T16:25:54+00:00,None,None,12.37.59,2026-07-09T21:14:35.116668+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""522c9be4-ab2a-48e1-98ba-d0b7e516..."
4157,339e0fb6e62933d64e201c92db8a028fbeb3ca131fc855...,google_play,com.google.android.apps.maps,Google Maps,bbab18a1-584d-4d90-aee9-94c37c4266a6,Maylay 1993,https://play-lh.googleusercontent.com/a/ACg8oc...,👍,5,0,2026-07-08T06:57:07+00:00,None,None,10.64.2,2026-07-09T21:14:55.165041+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""bbab18a1-584d-4d90-aee9-94c37c42..."
2530,cbf6ca9bba8d7ba280b03bb3b037e8ac5996a6417c3d12...,google_play,com.instagram.android,Instagram,90b57c73-ae05-4df0-8474-859190934cd6,Ashwin Puri,https://play-lh.googleusercontent.com/a/ACg8oc...,super,5,0,2026-07-08T18:54:53+00:00,None,None,None,2026-07-09T21:14:43.782074+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""90b57c73-ae05-4df0-8474-85919093..."
2784,c1a0f8601883571fac13680c65e566616736c7585d14bd...,google_play,com.instagram.android,Instagram,c35744de-4ec9-42ec-b103-8a8026dea0a5,Shazan Khan,https://play-lh.googleusercontent.com/a/ACg8oc...,my instagram account is disabled my account is...,1,0,2026-07-08T16:47:48+00:00,None,None,None,2026-07-09T21:14:43.782074+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""c35744de-4ec9-42ec-b103-8a8026de..."
4271,c2d87d52ed46b34c084b20a9bcc506bbf579cfa9ac69e8...,google_play,com.netflix.mediaclient,Netflix,e1823266-deb6-48de-bfca-0032e1dc6766,REHAN.M Rehan.,https://play-lh.googleusercontent.com/a-/ALV-U...,crazy bro,5,0,2026-07-08T08:56:22+00:00,None,None,None,2026-07-09T21:14:57.828674+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""e1823266-deb6-48de-bfca-0032e1dc..."
561,77221a5c8698cda914a43367b0ebafbfe4c0f1b161d0b3...,google_play,com.google.android.youtube,YouTube,a70cb5e1-7265-414f-a436-8c2baf72ee70,Fatima Md,https://play-lh.googleusercontent.com/a-/ALV-U...,"Great app, but regional payment and heavy ads ...",3,1,2026-07-08T15:24:27+00:00,None,None,21.26.364,2026-07-09T21:14:35.116668+00:00,phase2_cadence_runA_twice_daily_test_20260709_...,"{""reviewId"": ""a70cb5e1-7265-414f-a436-8c2baf72..."
1392,b22d7337cadc6d39c95b14fe531902d5fef6596be2ca88...,google_play,com.zhiliaoapp.musica

## 18. Save schema snapshot

In [20]:
schema_frames = []

for table_name in required_tables:
    table_info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    table_info.insert(0, "table_name", table_name)
    schema_frames.append(table_info)

schema_snapshot_df = pd.concat(schema_frames, ignore_index=True)

schema_snapshot_path = OUTPUT_DIR / "phase2_cadence_runA_database_schema_snapshot.csv"
schema_snapshot_df.to_csv(schema_snapshot_path, index=False)

print("Saved:", schema_snapshot_path)

display(schema_snapshot_df)

Saved: /content/outputs/phase2_cadence_runA_database_schema_snapshot.csv


,table_name,cid,name,type,notnull,dflt_value,pk
0,phase2_reviews_raw,0,review_key,TEXT,0,None,1
1,phase2_reviews_raw,1,source,TEXT,1,None,0
2,phase2_reviews_raw,2,app_id,TEXT,1,None,0
3,phase2_reviews_raw,3,app_name,TEXT,0,None,0
4,phase2_reviews_raw,4,review_id,TEXT,0,None,0
...,...,...,...,...,...,...,...
93,phase2_quality_flags,3,app_id,TEXT,0,None,0
94,phase2_quality_flags,4,flag_name,TEXT,0,None,0
95,phase2_quality_flags,5,flag_severity,TEXT,0,None,0
96,phase2_quality_flags,6,flag_value,TEXT,0,None,0


## 19. Create operating assumption findings report

This report directly answers the cadence and operating-model questions.

In [21]:
def fmt_pct(value):
    if value is None or pd.isna(value):
        return "N/A"
    return f"{value:.2%}"

latest_baseline = cadence_comparison_df[
    cadence_comparison_df["run_label"] == "phase2_day3_controlled_repeated_run"
].tail(1)

if len(latest_baseline) == 1:
    baseline_duplicate_rate = latest_baseline["duplicate_rate"].iloc[0]
    baseline_new_insert_rate = latest_baseline["new_insert_rate"].iloc[0]
    baseline_new_inserts = latest_baseline["new_records_inserted_total"].iloc[0]
    baseline_runtime = latest_baseline["runtime_seconds"].iloc[0]
    baseline_growth = latest_baseline["review_rows_growth"].iloc[0]
else:
    baseline_duplicate_rate = None
    baseline_new_insert_rate = None
    baseline_new_inserts = None
    baseline_runtime = None
    baseline_growth = None

runA_quality_flag_summary_df = pd.read_sql_query("""
    SELECT
        flag_name,
        flag_severity,
        COUNT(*) AS flag_count
    FROM phase2_quality_flags
    WHERE run_id = ?
    GROUP BY flag_name, flag_severity
    ORDER BY flag_count DESC
""", conn, params=(RUN_ID,))

top_new_apps = runA_app_ranking_df.sort_values(
    "new_records_inserted",
    ascending=False
).head(5)

highest_duplicate_apps = runA_app_ranking_df.sort_values(
    "duplicate_rate",
    ascending=False
).head(5)

slowest_apps = runA_app_ranking_df.sort_values(
    "runtime_seconds",
    ascending=False
).head(5)

findings_report = f"""# Phase 2 Cadence Test Run A Findings

## Purpose

This run starts the operating-assumption testing after the Day 1-Day 3 controlled repeated-run baseline.

The test compares the existing once-daily controlled baseline with a same-day second collection run.

## Run Setup

- Run label: `{RUN_LABEL}`
- Run ID: `{RUN_ID}`
- Frequency label: `{FREQUENCY_LABEL}`
- Source: Google Play
- Language / country: `{LANGUAGE}` / `{COUNTRY}`
- Apps: {len(app_config_df)}
- Target reviews per app: {TARGET_REVIEWS_PER_APP:,}
- Database: continued from the fixed Day 3 Phase 2 SQLite database

## Overall Run A Results

| Metric | Value |
|---|---:|
| Total fetched records | {records_fetched_total:,} |
| New records inserted | {new_records_inserted_total:,} |
| Duplicates skipped | {duplicates_skipped_total:,} |
| Duplicate rate | {fmt_pct(duplicate_rate)} |
| New insert rate | {fmt_pct(new_insert_rate)} |
| Runtime seconds | {collection_runtime_seconds:.2f} |
| Runtime minutes | {collection_runtime_seconds / 60:.2f} |
| Raw review rows before | {raw_rows_before:,} |
| Raw review rows after | {raw_rows_after:,} |
| Raw review row growth | {review_rows_growth:,} |
| Database size before | {db_size_before_mb:.2f} MB |
| Database size after | {db_size_after_mb:.2f} MB |
| Database size growth | {db_size_growth_mb:.2f} MB |
| Errors | {errors_total} |
| Quality flags | {quality_flag_total:,} |

## Once-Daily Baseline vs. Twice-Daily Run A

| Metric | Latest once-daily baseline | Twice-daily Run A |
|---|---:|---:|
| New records inserted | {baseline_new_inserts if baseline_new_inserts is not None else "N/A"} | {new_records_inserted_total:,} |
| Duplicate rate | {fmt_pct(baseline_duplicate_rate)} | {fmt_pct(duplicate_rate)} |
| New insert rate | {fmt_pct(baseline_new_insert_rate)} | {fmt_pct(new_insert_rate)} |
| Runtime seconds | {baseline_runtime if baseline_runtime is not None else "N/A"} | {collection_runtime_seconds:.2f} |
| Review row growth | {baseline_growth if baseline_growth is not None else "N/A"} | {review_rows_growth:,} |

## Run A App-Level Results

{runA_app_ranking_df[[
    "app_name",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "new_insert_rate",
    "runtime_seconds",
    "quality_flag_count",
    "max_review_date"
]].to_markdown(index=False)}

## Apps With the Most New Inserts

{top_new_apps[[
    "app_name",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "new_insert_rate",
    "max_review_date"
]].to_markdown(index=False)}

## Apps With the Highest Duplicate Rate

{highest_duplicate_apps[[
    "app_name",
    "new_records_inserted",
    "duplicates_skipped",
    "duplicate_rate",
    "quality_flag_count"
]].to_markdown(index=False)}

## Slowest Apps

{slowest_apps[[
    "app_name",
    "runtime_seconds",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "quality_flag_count"
]].to_markdown(index=False)}

## Quality Flag Summary

"""

if len(runA_quality_flag_summary_df) > 0:
    findings_report += runA_quality_flag_summary_df.to_markdown(index=False)
else:
    findings_report += "No quality flags were created in this run."

findings_report += """

## Operating Assumption Notes

### Once-daily vs. twice-daily collection

This run provides the first same-day second-collection result. It should be compared with the Day 1-Day 3 controlled repeated baseline, especially Day 3, because Day 3 is the latest once-daily-style baseline run before cadence testing.

### High-activity vs. lower-activity apps

App-level new inserts and new insert rates show whether some apps are producing meaningfully more newly captured reviews than others under the same target and source setup.

### Duplicate rate under higher frequency

The duplicate rate from Run A indicates whether a same-day second collection starts producing too many repeated records relative to new captures.

### Runtime and database growth

Runtime, database growth in MB, and review row growth show whether the higher collection frequency is still operationally reasonable.

### App-specific instability or quality-flag patterns

App-level errors, runtime, and quality flags help identify whether any app is unstable or producing unusual data-quality behavior.

## Next Step

Run at least one more cadence test later under the same setup before making a final recommendation on once-daily vs. twice-daily collection.
"""

findings_report_path = OUTPUT_DIR / "phase2_cadence_operating_assumption_findings.md"
findings_report_path.write_text(findings_report, encoding="utf-8")

print("Saved:", findings_report_path)
print(findings_report[:3000])

Saved: /content/outputs/phase2_cadence_operating_assumption_findings.md
# Phase 2 Cadence Test Run A Findings

## Purpose

This run starts the operating-assumption testing after the Day 1-Day 3 controlled repeated-run baseline.

The test compares the existing once-daily controlled baseline with a same-day second collection run.

## Run Setup

- Run label: `phase2_cadence_runA_twice_daily_test`
- Run ID: `phase2_cadence_runA_twice_daily_test_20260709_210858`
- Frequency label: `twice_daily_same_day_second_run`
- Source: Google Play
- Language / country: `en` / `us`
- Apps: 10
- Target reviews per app: 1,200
- Database: continued from the fixed Day 3 Phase 2 SQLite database

## Overall Run A Results

| Metric | Value |
|---|---:|
| Total fetched records | 12,000 |
| New records inserted | 4,395 |
| Duplicates skipped | 7,605 |
| Duplicate rate | 63.38% |
| New insert rate | 36.62% |
| Runtime seconds | 28.14 |
| Runtime minutes | 0.47 |
| Raw review rows before | 17,815 |
| Raw review ro

## 20. Compress the updated SQLite database

In [22]:
compressed_db_path = OUTPUT_DIR / "google_play_reviews_after_cadence_runA.sqlite.zip"

if compressed_db_path.exists():
    compressed_db_path.unlink()

with zipfile.ZipFile(compressed_db_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(DB_PATH, arcname=DB_PATH.name)

compressed_db_size_mb = compressed_db_path.stat().st_size / (1024 * 1024)

print("Saved compressed database:")
print(compressed_db_path)
print(f"Compressed database size: {compressed_db_size_mb:.2f} MB")

Saved compressed database:
/content/outputs/google_play_reviews_after_cadence_runA.sqlite.zip
Compressed database size: 13.77 MB


## 21. Create output file manifest

In [23]:
output_files = sorted([p for p in OUTPUT_DIR.glob("*") if p.is_file()])

file_manifest_df = pd.DataFrame([
    {
        "file_name": p.name,
        "path": str(p),
        "size_mb": p.stat().st_size / (1024 * 1024),
    }
    for p in output_files
])

file_manifest_path = OUTPUT_DIR / "phase2_cadence_runA_output_file_manifest.csv"
file_manifest_df.to_csv(file_manifest_path, index=False)

print("Saved:", file_manifest_path)

display(file_manifest_df)

Saved: /content/outputs/phase2_cadence_runA_output_file_manifest.csv


,file_name,path,size_mb
0,google_play_reviews_after_cadence_runA.sqlite.zip,/content/outputs/google_play_reviews_after_cad...,13.767809
1,phase2_cadence_app_comparison_through_runA.csv,/content/outputs/phase2_cadence_app_comparison...,0.010741
2,phase2_cadence_comparison_through_runA.csv,/content/outputs/phase2_cadence_comparison_thr...,0.001450
3,phase2_cadence_operating_assumption_findings.md,/content/outputs/phase2_cadence_operating_assu...,0.007573
4,phase2_cadence_runA_app_level_summary.csv,/content/outputs/phase2_cadence_runA_app_level...,0.002339
5,phase2_cadence_runA_app_new_insert_ranking.csv,/content/outputs/phase2_cadence_runA_app_new_i...,0.002671
6,phase2_cadence_runA_database_schema_snapshot.csv,/content/outputs/phase2_cadence_runA_database_...,0.004558
7,phase2_cadence_runA_quality_flags.csv,/content/outputs/phase2_cadence_runA_quality_f...,3.307288
8,phase2_cadence_runA_run_summary.csv,/content/outputs/phase2_cadence_runA_run_summa...,0.000958
9,phase2_cadence_runA_sample_new_reviews.csv,/content/outputs/phase2_cadence_runA_sample_ne...,0.190169


## 22. Final validation checklist

In [24]:
final_run_history_count = len(cadence_comparison_df)

checks = {
    "same_10_apps": len(app_config_df) == 10,
    "same_1200_review_target": TARGET_REVIEWS_PER_APP == 1200,
    "continued_from_fixed_day3_database": raw_rows_before >= 17000 and prior_run_count >= 3,
    "phase2_tables_used": all(t in existing_tables for t in required_tables),
    "cadence_run_record_saved": len(run_summary_df) == 1,
    "cadence_app_summary_has_10_rows": len(app_summary_df) == 10,
    "records_fetched_total_available": records_fetched_total >= 0,
    "new_insert_total_available": new_records_inserted_total >= 0,
    "duplicates_skipped_total_available": duplicates_skipped_total >= 0,
    "database_growth_matches_new_inserts": review_rows_growth == new_records_inserted_total,
    "cleaned_table_growth_matches_new_inserts": (cleaned_rows_after - cleaned_rows_before) == new_records_inserted_total,
    "history_includes_day1_day2_day3_and_runA": final_run_history_count >= 4,
    "runtime_available": collection_runtime_seconds > 0,
    "run_summary_saved": run_summary_path.exists(),
    "app_summary_saved": app_summary_path.exists(),
    "quality_flags_saved": quality_flags_path.exists(),
    "cadence_comparison_saved": cadence_comparison_path.exists(),
    "cadence_app_comparison_saved": cadence_app_comparison_path.exists(),
    "sample_new_reviews_saved": sample_new_reviews_path.exists(),
    "schema_snapshot_saved": schema_snapshot_path.exists(),
    "findings_report_saved": findings_report_path.exists(),
    "compressed_database_saved": compressed_db_path.exists(),
    "file_manifest_saved": file_manifest_path.exists(),
}

check_df = pd.DataFrame([
    {"check": check_name, "passed": passed}
    for check_name, passed in checks.items()
])

display(check_df)

if not all(checks.values()):
    failed_checks = [check_name for check_name, passed in checks.items() if not passed]
    raise ValueError(f"Some final validation checks failed: {failed_checks}")

print("All final validation checks passed.")

,check,passed
0,same_10_apps,True
1,same_1200_review_target,True
2,continued_from_fixed_day3_database,True
3,phase2_tables_used,True
4,cadence_run_record_saved,True
5,cadence_app_summary_has_10_rows,True
6,records_fetched_total_available,True
7,new_insert_total_available,True
8,duplicates_skipped_total_available,True
9,database_growth_matches_new_inserts,True


All final validation checks passed.


## 23. Create and download GitHub upload zip

In [25]:
zip_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
github_zip_path = Path("/content") / f"phase2_cadence_runA_github_upload_files_{zip_timestamp}.zip"

files_to_include = [
    OUTPUT_DIR / "phase2_cadence_runA_run_summary.csv",
    OUTPUT_DIR / "phase2_cadence_runA_app_level_summary.csv",
    OUTPUT_DIR / "phase2_cadence_runA_quality_flags.csv",
    OUTPUT_DIR / "phase2_cadence_comparison_through_runA.csv",
    OUTPUT_DIR / "phase2_cadence_app_comparison_through_runA.csv",
    OUTPUT_DIR / "phase2_cadence_runA_app_new_insert_ranking.csv",
    OUTPUT_DIR / "phase2_cadence_runA_sample_new_reviews.csv",
    OUTPUT_DIR / "phase2_cadence_runA_database_schema_snapshot.csv",
    OUTPUT_DIR / "phase2_cadence_operating_assumption_findings.md",
    OUTPUT_DIR / "phase2_cadence_runA_output_file_manifest.csv",
    OUTPUT_DIR / "google_play_reviews_after_cadence_runA.sqlite.zip",
]

if github_zip_path.exists():
    github_zip_path.unlink()

with zipfile.ZipFile(github_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in files_to_include:
        if file_path.exists():
            zf.write(file_path, arcname=f"outputs/{file_path.name}")
        else:
            print("Missing file, not included:", file_path)

print("GitHub upload zip created:")
print(github_zip_path)
print(f"Zip size: {github_zip_path.stat().st_size / (1024 * 1024):.2f} MB")

files.download(str(github_zip_path))

GitHub upload zip created:
/content/phase2_cadence_runA_github_upload_files_20260709_211803_utc.zip
Zip size: 14.78 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cadence Run A conclusion

Cadence Run A completed the first same-day second-collection test after the Day 1-Day 3 controlled repeated-run baseline.

The setup kept the same 10 apps and the same 1,200-review target, continued from the fixed Day 3 SQLite database, inserted only new review IDs, skipped existing duplicates, and saved run-level, app-level, quality-flag, schema, comparison, findings, sample output, and compressed database files.

This run can now be used to evaluate whether twice-daily collection produces enough new reviews to justify the higher duplicate rate, runtime, and database growth compared with the once-daily controlled baseline.